In [ ]:
import json
import pandas as pd
from tqdm import tqdm
from datasets import load_dataset, Dataset, concatenate_datasets
from openai import OpenAI
import time
import os

CODE_DATASET_PATH = "businessrules/Code_snippets"
RULE_DATASET_PATH = "businessrules/100_5algorithms"

OUTPUT_DATASET_PATH = "businessrules/judge_5_algorithm_3"

In [ ]:

CODE_COLUMN_NAME = "txt_file"
CODE_ID_COLUMN = "ID"
RESULTS_ID_COLUMN = "ID"


SAVE_PATH = "/content/drive/MyDrive/full_evaluation_results7.csv"
SAVE_EVERY = 10  

In [ ]:
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")

client = OpenAI(
    api_key=openrouter_api_key,
    base_url="https://openrouter.ai/api/v1"
)

import os
os.environ["HF_TOKEN"] = "hf_token" # removed for safety

from huggingface_hub import login
login(os.environ["HF_TOKEN"])

LLM_MODELS = {
    "A": "google/gemini-2.5-flash",
    "B": "anthropic/claude-3.7-sonnet",
    "C": "openai/gpt-4o",
    "D": "openai/gpt-5"
}

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [ ]:
from google.colab import drive

print("Mounting Google Drive...")
drive.mount('/content/drive')

In [ ]:
SYSTEM_INSTRUCTION = """
SYSTEM INSTRUCTION: BUSINESS RULE QUALITY JUDGE

ROLE:
You are an expert Business Analyst tasked with acting as a rigorous, objective judge.
Your sole function is to evaluate the quality, completeness, and clarity of a set of Business Rules (BRs)
provided in a Markdown document (RULES_MD) against a corresponding block of source code (CODE).

PRIORITY:
You must adhere strictly to the defined rubrics below and your final output MUST be in the specified JSON format.

## CORE DEFINITIONS

**Definition of a Business Rule (BR):**
A BR is a constraint, condition, or policy that dictates how the business operates.
A BR focuses on the *WHAT* (policy, condition, or constraint) — not the *HOW* (technical implementation or code structure).

## FORBIDDEN TECHNICAL TERMS
Use of the following terms *in the RULES_MD* indicates technical bias and must lower the score under `technical_jargon_rate`:
API, service, command, UI, database, schema, cache, deployment, thread, framework, library, middleware, endpoint,
logging, environment, configuration, instance, repository, controller, model, function, variable.

However:
- If these terms are used in *business contexts* (e.g., "customer service", "payment command"), they are acceptable.
- Only software/implementation usage should reduce the score.

## EVALUATION PROCESS

Before scoring, you must reason internally (but output only the JSON):
1. Identify all business rules, policies, and constraints explicitly expressed or implied in the `CODE`.
2. Identify all business rules listed in the `RULES_MD`.
3. Compare them based on the seven rubrics below.
4. Score each criterion independently and objectively.

## EVALUATION RUBRIC (SEVEN METRICS)

### 1. business_rule_percentage (RELEVANCE) — 0–100
**Goal:** Measure how much of the `RULES_MD` content represents true business rules (policies/constraints).
**How to score:**
- Calculate this percentage **by statement**.
- `(Number of statements that are true business rules) / (Total number of content statement in RULES_MD, excluding empty lines/headers) * 100`.
- **100:** Every statement describes a business policy or constraint.
- **0:** All statement describe technical details or implementation.
**Special cases:**
- If `RULES_MD` is empty → 0
- If `RULES_MD` says “no business rules” and the code truly contains none → 100
- If `RULES_MD` says “no business rules” but the code clearly contains business logic → 0

### 2. format_readability_rate (READABILITY) — 1–5
**Goal:** Evaluate structure, clarity, and *layout*.
**How to score:**
- **5:** Professionally formatted: uses Markdown (headings, lists) effectively, groups related rules logically.
- **3:** Readable, but messy. Poor/no use of Markdown, or no logical grouping.
- **1:** A single, unstructured wall of text or extremely confusing layout.
**Special cases:**
- Empty `RULES_MD` → 0
- Correctly written “no rules” explanation → 5

### 3. technical_jargon_rate (JARGON AVOIDANCE) — 1–5
**Goal:** Measure how well the `RULES_MD` avoids technical terminology.
**Critical Instruction:** This score evaluates **only** the text inside `RULES_MD`. It does *not* matter if the `CODE` contains these terms. Penalize only the *rules* for using implementation language.
**How to score:**
- **5:** Purely business language; zero forbidden words and technical terms used in a technical context.
- **3:** 1-2 forbidden terms and technical words are used, or the phrasing implies technical knowledge (e.g., "calls the function").
- **1:** Heavy technical jargon (3+ forbidden terms); reads like a technical spec.
**Special cases:**
- If `RULES_MD` correctly claims there are no business rules and this matches `CODE` → 5
- If `RULES_MD` is empty → 5

---

### 4. coverage_rate (COMPLETENESS) — 1–5
**Goal:** Measure how *completely* the `RULES_MD` captures all business logic, policies, and constraints found in the `CODE`. This metric checks for **missing rules**.
**How to score:**
1. Read the `CODE` and identify all distinct business logic paths (e.g., validations, conditions, constraints, transformations).
2. Check if `RULES_MD` describes them.
3. Score based on completeness:
    - **5:** 100% of business logic paths in the `CODE` are described in `RULES_MD`.
    - **4:** Covers the main success path and most constraints (>80%), but misses one or two minor edge cases.
    - **3:** Covers the main success path but misses significant constraints or error paths (approx. 50-80% coverage).
    - **1:** Misses major business logic; describes <50% of the policies in the code.
**Special cases:**
- If `RULES_MD` claims no rules and code indeed contains none → 5
- If `RULES_MD` claims no rules but code clearly contains business constraints → 1

---

### 5. testability_rate (PRECISION & CLARITY) — 1–5
**Goal:** Evaluate how precisely the rules are written as *verifiable conditions and outcomes*.
**How to score:**
- **5:** All rules are precise, unambiguous, and written as testable conditions (e.g., "IF [condition] THEN [outcome]").
- **3:** Rules describe behavior but are vague or not written as explicit conditions (e.g., "The system validates the user.").
- **1:** Rules are high-level, untestable intentions (e.g., "We must ensure good data.").
**Special case:** If `RULES_MD` says “no business rules” and code has none → 5.

---

### 6. atomicity_rate (STRUCTURE & REDUNDANCY) — 1–5
**Goal:** Check whether each rule expresses a single, distinct idea (no merging or repetition).
**How to score:**
- **5:** Each rule is atomic (expresses one idea) and non-redundant (not repeated).
- **3:** Some rules (approx. 10-30%) are bundled (merge multiple ideas) or are redundant.
- **1:** Highly repetitive, or most rules (>30%) are bundled.
**Special case:** If `RULES_MD` says “no business rules” and code has none → 5.

---

### 7. faithfulness_rate (ACCURACY) — 1–5
**Goal:** Evaluate how *accurately* each rule in `RULES_MD` reflects the `CODE`. This metric checks for **hallucinations or misrepresentations**.
**How to score:**
1. Read each individual rule in `RULES_MD`.
2. Find the corresponding logic in the `CODE`.
3. Score based on accuracy:
    - **5:** Every rule in `RULES_MD` is a precise, accurate, and non-exaggerated description of logic found in the `CODE`.
    - **3:** Most rules are accurate, but 1-2 rules are slightly misaligned, inferred, or misrepresent the code's behavior.
    - **1:** Multiple rules in `RULES_MD` are false, contradicted by the code, or completely fabricated (hallucinated) and do not exist in the code.
**Special cases:**
- If `RULES_MD` says "no rules" and that is accurate → 5.
- If `RULES_MD` says "no rules" but there *are* rules → 1.

## OUTPUT REQUIREMENTS

You MUST return a **single valid JSON object** following this exact schema.
Do not include any introductory or concluding text outside this JSON block.

{
  "business_rule_percentage": [integer 0-100],
  "format_readability_rate": [integer 1-5],
  "technical_jargon_rate": [integer 1-5],
  "coverage_rate": [integer 1-5],
  "testability_rate": [integer 1-5],
  "atomicity_rate": [integer 1-5],
  "faithfulness_rate": [integer 1-5],
}


## INPUT FORMAT

CODE:
[The source code block]

RULES_MD:
[The markdown document containing the business rules]
"""


In [ ]:
def evaluate_pair(model_name: str, code: str, rules_md: str) -> dict:
    """
    Sends a single (code, rule) pair to the LLM and returns parsed JSON output.
    Assumes the LLM returns a flat JSON object with 7 numeric scores.
    """
    score_keys = [
        "business_rule_percentage",
        "format_readability_rate",
        "technical_jargon_rate",
        "coverage_rate",
        "testability_rate",
        "atomicity_rate",
        "faithfulness_rate"
    ]

    try:
        response = client.chat.completions.create(
            model=model_name,
            response_format={"type": "json_object"},
            messages=[
                {"role": "system", "content": SYSTEM_INSTRUCTION},
                {"role": "user", "content": f"CODE:\n{code}\n\nRULES_MD:\n{rules_md}"}
            ],
            temperature=0
        )

        raw_output = response.choices[0].message.content.strip()
        result = json.loads(raw_output)

        final_scores = {}
        for key in score_keys:
            final_scores[key] = result.get(key)

        final_scores["error"] = None
        return final_scores

    except Exception as e:
        print(f"Error evaluating '{model_name}': {e}")
        error_result = {key: None for key in score_keys}
        error_result["error"] = str(e)
        return error_result

In [ ]:
def evaluate_pair(model_name: str, code: str, rules_md: str) -> dict:
    """
    Sends a single (code, rule) pair to the LLM and returns parsed JSON output.
    Assumes the LLM returns a flat JSON object with 7 numeric scores.
    Includes debug prints for troubleshooting.
    """
    score_keys = [
        "business_rule_percentage",
        "format_readability_rate",
        "technical_jargon_rate",
        "coverage_rate",
        "testability_rate",
        "atomicity_rate",
        "faithfulness_rate"
    ]

    try:
        print(f"\n--- Evaluating with {model_name} ---")

        response = client.chat.completions.create(
            model=model_name,
            response_format={"type": "json_object"},
            messages=[
                {"role": "system", "content": SYSTEM_INSTRUCTION},
                {"role": "user", "content": f"CODE:\n{code}\n\nRULES_MD:\n{rules_md}"}
            ],
            temperature=0
        )

        if not response or not hasattr(response, "choices") or not response.choices:
            raise ValueError(f"No valid response returned from {model_name}: {response}")

        raw_output = response.choices[0].message.content
        if raw_output is None:
            raise ValueError(f"{model_name} returned None as message content.")
        raw_output = raw_output.strip()

        print(f"Raw output from {model_name}:\n{repr(raw_output[:300])}")
        if len(raw_output) == 0:
            raise ValueError(f"{model_name} returned an empty string.")

        try:
            result = json.loads(raw_output)
        except json.JSONDecodeError as je:
            print(f" JSON decoding failed for {model_name}: {je}")
            print(f" Raw output (first 500 chars): {raw_output[:500]}")
            raise

        final_scores = {key: result.get(key) for key in score_keys}
        final_scores["error"] = None

        return final_scores

    except Exception as e:
        print(f" Error evaluating '{model_name}': {e}")
        error_result = {key: None for key in score_keys}
        error_result["error"] = str(e)
        return error_result


In [ ]:
import json, re

def evaluate_pair(model_name: str, code: str, rules_md: str) -> dict:
    """
    Sends a single (code, rule) pair to the LLM and returns parsed JSON output.
    Handles models that return Markdown-wrapped JSON (like Gemini),
    and gracefully reports parsing errors.
    """
    score_keys = [
        "business_rule_percentage",
        "format_readability_rate",
        "technical_jargon_rate",
        "coverage_rate",
        "testability_rate",
        "atomicity_rate",
        "faithfulness_rate"
    ]

    try:
        use_json_format = not any(x in model_name.lower() for x in ["gemini"])
        response_format = {"type": "json_object"} if use_json_format else None

        response = client.chat.completions.create(
            model=model_name,
            response_format=response_format,
            messages=[
                {"role": "system", "content": SYSTEM_INSTRUCTION},
                {"role": "user", "content": f"CODE:\n{code}\n\nRULES_MD:\n{rules_md}"}
            ],
            temperature=0
        )

        if not response or not hasattr(response, "choices") or not response.choices:
            raise ValueError(f"No valid response returned from {model_name}: {response}")

        raw_output = response.choices[0].message.content
        if raw_output is None:
            raise ValueError(f"{model_name} returned None as message content.")
        raw_output = raw_output.strip()

        if raw_output.startswith("```"):
            raw_output = re.sub(r"^```[a-zA-Z]*\n?", "", raw_output)
            raw_output = raw_output.replace("```", "").strip()

        try:
            result = json.loads(raw_output)
        except json.JSONDecodeError as je:
            print(f" JSON decoding failed for {model_name}: {je}")
            print(f" Raw output (first 500 chars):\n{raw_output[:500]}")
            raise

        final_scores = {key: result.get(key) for key in score_keys}
        final_scores["error"] = None
        return final_scores

    except Exception as e:
        print(f" Error evaluating '{model_name}': {e}")
        error_result = {key: None for key in score_keys}
        error_result["error"] = str(e)
        return error_result


In [ ]:
def run_evaluation():
    # --- 1. Load both datasets ---
    print("Loading datasets...")
    code_ds = load_dataset(CODE_DATASET_PATH)["train"]
    results_ds = load_dataset(RULE_DATASET_PATH)["train"]

    # --- 2. Convert to pandas and normalize ID types ---
    code_df = code_ds.to_pandas()
    results_df = results_ds.to_pandas()

    print(f"Loaded {len(code_df)} code snippets and {len(results_df)} result rows.")

    code_df[CODE_ID_COLUMN] = code_df[CODE_ID_COLUMN].astype(str)
    results_df[RESULTS_ID_COLUMN] = results_df[RESULTS_ID_COLUMN].astype(str)

    code_df.rename(columns={CODE_ID_COLUMN: "ID"}, inplace=True)
    results_df.rename(columns={RESULTS_ID_COLUMN: "ID"}, inplace=True)

    # --- 3. Merge on ID ---
    merged_df = pd.merge(code_df, results_df, on="ID", how="inner")
    merged_df = merged_df[merged_df["ID"].astype(int).between(89, 100)]
    print(f"Successfully merged. {len(merged_df)} matching IDs found.")

    # --- 4. Identify the 'result_i' columns ---
    result_cols = [col for col in merged_df.columns if col.startswith('result_')]
    if not result_cols:
        raise ValueError("No columns found starting with 'result_' in the results dataset.")
    print(f"Found {len(result_cols)} result columns to evaluate: {result_cols}")

    # --- 5. Load existing results to avoid re-work ---
    all_evaluations = []
    completed_evals = set()

    if os.path.exists(SAVE_PATH):
        print(f"Found existing results at {SAVE_PATH}. Loading to resume...")
        try:
            existing_df = pd.read_csv(SAVE_PATH)
            all_evaluations = existing_df.to_dict('records')

            for eval_row in all_evaluations:
                eval_key = (
                    str(eval_row.get('code_id')),
                    eval_row.get('result_source'),
                    eval_row.get('judging_llm')
                )
                completed_evals.add(eval_key)
            print(f"Loaded {len(all_evaluations)} existing evaluations. Will skip these.")
        except pd.errors.EmptyDataError:
            print("Found empty results file. Starting from scratch.")
        except Exception as e:
            print(f"Error loading existing results: {e}. Starting from scratch.")
            all_evaluations = [] 
            completed_evals = set()
    else:
        print("No existing results file found. Starting from scratch.")

    # --- 6. Iterate, Evaluate, and Create Long-Format Data ---
    all_evaluations = []
    total_evals = len(merged_df) * len(result_cols) * len(LLM_MODELS)
    print(f"Starting evaluation... Total evaluations to perform: {total_evals}")

    pbar = tqdm(total=total_evals, desc="Evaluating all pairs")
    pbar.update(len(completed_evals))

    try:
        for _, row in merged_df.iterrows():
            code_id = row["ID"]
            code = row.get(CODE_COLUMN_NAME)

            if not code:
                print(f"Skipping ID {code_id}: Missing code.")
                continue

            for res_col_name in result_cols:
                rules = row.get(res_col_name)

                if not rules or pd.isna(rules):
                    print(f"Skipping ID {code_id}, {res_col_name}: Missing rules.")
                    pbar.update(len(LLM_MODELS))
                    continue

                for llm_label, model_name in LLM_MODELS.items():

                    eval_key = (str(code_id), res_col_name, llm_label)
                    if eval_key in completed_evals:
                        continue 

       
                    try:
                      print(f"\n--- Evaluating {code_id} | {res_col_name} | {llm_label} ({model_name}) ---")
                      scores = evaluate_pair(model_name, code, rules)
                    except Exception as e:
                      print(f" Error evaluating '{model_name}' for code_id={code_id}, result={res_col_name}: {e}")

                    result_row = {
                        "code_id": code_id,
                        "result_source": res_col_name, 
                        "judging_llm": llm_label,    
                    }
                    result_row.update(scores) 

                    all_evaluations.append(result_row)
                    pbar.update(1) 

                    if len(all_evaluations) % SAVE_EVERY == 0:
                        print(f"\n Saving checkpoint to {SAVE_PATH}...")
                        pd.DataFrame(all_evaluations).to_csv(SAVE_PATH, index=False)

                    time.sleep(0.5) 

    except KeyboardInterrupt:
        print("\nProcess interrupted. Saving partial results...")
    finally:
        pbar.close()

    # --- 7. Final Save & Upload ---
    if not all_evaluations:
        print("No evaluations were performed.")
        return None

    print("Evaluation complete. Converting to DataFrame...")
    final_df = pd.DataFrame(all_evaluations)

    print(f" Saving final results to {SAVE_PATH}...")
    final_df.to_csv(SAVE_PATH, index=False)

    # --- 8. Push to Hugging Face Hub ---
    print(f" Uploading to {OUTPUT_DATASET_PATH} on HF Hub...")
    ds = Dataset.from_pandas(final_df)
    ds.push_to_hub(OUTPUT_DATASET_PATH, private=True) 

    print(" All done.")
    return final_df


if __name__ == "__main__":
    final_results_df = run_evaluation()

    if final_results_df is not None:
        print("\n--- Evaluation Summary ---")
        print(final_results_df.head())
        print(f"\nTotal evaluations: {len(final_results_df)}")
        print("\nScore columns:")
        print(final_results_df.columns)

In [ ]:
import pandas as pd
import os


SAVE_PATH = "/content/drive/MyDrive/full_evaluation_results6.csv"

SCORE_COLUMNS = [
    "business_rule_percentage",
    "format_readability_rate",
    "technical_jargon_rate",
    "coverage_rate",
    "testability_rate",
    "atomicity_rate",
    "faithfulness_rate"
]

def generate_evaluation_report(start_id=1, end_id=88):
    """
    Loads the evaluation results, filters by code ID range, and calculates
    the average of each score metric grouped by code_id and result_source.
    """
    if not os.path.exists(SAVE_PATH):
        print(f"Error: Results file not found at {SAVE_PATH}.")
        print("Please ensure Google Drive is mounted and the file exists.")
        return None

    print(f"Loading data from {SAVE_PATH}...")
    df = pd.read_csv(SAVE_PATH)


    try:
        df['code_id'] = df['code_id'].astype(int)
    except Exception:
        print("Warning: Could not convert 'code_id' column to integer. Skipping filtering.")

    print(f"Filtering data for code IDs {start_id} through {end_id}...")
    df_filtered = df[df['code_id'].between(start_id, end_id)]

    if df_filtered.empty:
        print("Filtered DataFrame is empty. Check your file contents or ID range.")
        return None


    print("Aggregating results by Code ID and Result Source...")
    df_report = df_filtered.groupby(['code_id', 'result_source'])[SCORE_COLUMNS].mean().reset_index()

    df_report[SCORE_COLUMNS] = df_report[SCORE_COLUMNS].round(2)

    return df_report

if __name__ == "__main__":


    report = generate_evaluation_report(start_id=1, end_id=88)

    if report is not None:
        print("\n" + "="*80)
        print("AGGREGATED EVALUATION REPORT (Code IDs 1-48, Average Scores per Result Source)")
        print("="*80)
        pd.set_option('display.max_rows', None)
        pd.set_option('display.max_columns', None)
        pd.set_option('display.width', 1000)
        print(report)
        print("="*80)
        print(f"Total aggregated rows: {len(report)}")

Loading data from /content/drive/MyDrive/full_evaluation_results6.csv...
Filtering data for code IDs 1 through 88...
Aggregating results by Code ID and Result Source...

AGGREGATED EVALUATION REPORT (Code IDs 1-48, Average Scores per Result Source)
     code_id result_source  business_rule_percentage  format_readability_rate  technical_jargon_rate  coverage_rate  testability_rate  atomicity_rate  faithfulness_rate
0          1      result_1                    100.00                     5.00                   5.00           4.75              5.00            5.00               4.75
1          1      result_2                    100.00                     5.00                   5.00           4.75              5.00            4.75               4.75
2          1      result_3                    100.00                     5.00                   5.00           4.50              5.00            5.00               4.75
3          1      result_4                    100.00                     5.

In [ ]:
import pandas as pd
import os


SAVE_PATH = "/content/drive/MyDrive/full_evaluation_results6.csv"

SCORE_COLUMNS = [
    "business_rule_percentage",
    "format_readability_rate",
    "technical_jargon_rate",
    "coverage_rate",
    "testability_rate",
    "atomicity_rate",
    "faithfulness_rate"
]

def generate_evaluation_report(start_id=1, end_id=100):
    """
    Loads the evaluation results, filters by code ID range, and calculates
    the average of each score metric grouped by code_id and result_source.
    Returns the aggregated DataFrame for further analysis.
    """
    if not os.path.exists(SAVE_PATH):
        print(f"Error: Results file not found at {SAVE_PATH}.")
        print("Please ensure Google Drive is mounted and the file exists.")
        return None

    print(f"Loading data from {SAVE_PATH}...")
    df = pd.read_csv(SAVE_PATH)


    try:
        df['code_id'] = df['code_id'].astype(int)
    except Exception:
        print("Warning: Could not convert 'code_id' column to integer. Skipping filtering.")

    print(f"Filtering data for code IDs {start_id} through {end_id}...")
    df_filtered = df[df['code_id'].between(start_id, end_id)]

    if df_filtered.empty:
        print("Filtered DataFrame is empty. Check your file contents or ID range.")
        return None

    print("Aggregating results by Code ID and Result Source...")
    df_report = df_filtered.groupby(['code_id', 'result_source'])[SCORE_COLUMNS].mean().reset_index()


    return df_report

def generate_summary_report(df_aggregated):
    """
    Takes the report aggregated by code_id and result_source, and further
    aggregates it to show the grand mean for each result_source (result_1, etc.)
    """
    if df_aggregated.empty:
        return None

    print("\nCalculating GRAND SUMMARY by Result Source...")

    df_summary = df_aggregated.groupby('result_source')[SCORE_COLUMNS].mean().reset_index()

    df_summary[SCORE_COLUMNS] = df_summary[SCORE_COLUMNS].round(2)

    return df_summary


if __name__ == "__main__":


    # Step 1: Generate the code-level report (intermediate step)
    intermediate_report = generate_evaluation_report(start_id=1, end_id=100)

    if intermediate_report is not None:
        # Step 2: Generate the summary report based on the intermediate data
        summary_report = generate_summary_report(intermediate_report)

        if summary_report is not None:
            print("\n" + "="*80)
            print("GRAND SUMMARY REPORT (Average Performance of Each Result Type, IDs 1-100)")
            print("="*80)
            pd.set_option('display.max_rows', None)
            pd.set_option('display.max_columns', None)
            pd.set_option('display.width', 1000)
            print(summary_report)
            print("="*80)
            print(f"Total result types summarized: {len(summary_report)}")

Loading data from /content/drive/MyDrive/full_evaluation_results6.csv...
Filtering data for code IDs 1 through 100...
Aggregating results by Code ID and Result Source...

Calculating GRAND SUMMARY by Result Source...

GRAND SUMMARY REPORT (Average Performance of Each Result Type, IDs 1-100)
  result_source  business_rule_percentage  format_readability_rate  technical_jargon_rate  coverage_rate  testability_rate  atomicity_rate  faithfulness_rate
0      result_1                     98.01                     5.00                   4.89           4.59              4.56            4.84               4.66
1      result_2                     97.48                     4.99                   4.87           4.56              4.50            4.81               4.60
2      result_3                     97.14                     5.00                   4.89           4.55              4.50            4.78               4.64
3      result_4                     98.86                     5.00          

Max score = 30

- ***result 1: Score = 28.54 ...... percentage = 98.01%***

- result 2: Score = 28.33 ...... percentage = 97.48%

- result 3: Score = 28.36 ...... percentage = 97.14%

- ***result 4: Score = 28.50 ...... percentage = 98.86%***

- result 5: Score = 28.48 ...... percentage = 98.23%

Without GPT 5

Max score = 30

- result 1: Score = 29.16 ...... percentage = 98.48%

- result 2: Score = 28.96 ...... percentage = 98.40%

- result 3: Score = 28.99 ...... percentage = 98.14%

- result 4: Score = 29.13 ...... percentage = 99.11%

- ***result 5: Score = 29.2 ...... percentage = 99.28%***




In [ ]:
import pandas as pd
import os

# --- CONSTANTS ---

SAVE_PATH = "/content/drive/MyDrive/full_evaluation_results6.csv"

SCORE_COLUMNS = [
    "business_rule_percentage",
    "format_readability_rate",
    "technical_jargon_rate",
    "coverage_rate",
    "testability_rate",
    "atomicity_rate",
    "faithfulness_rate"
]

def generate_evaluation_report(start_id=1, end_id=100):
    """
    Loads the evaluation results, filters by code ID range, and calculates
    the average of each score metric grouped by code_id and result_source.
    Returns the aggregated DataFrame for further analysis.
    """
    if not os.path.exists(SAVE_PATH):
        print(f"Error: Results file not found at {SAVE_PATH}.")
        print("Please ensure Google Drive is mounted and the file exists.")
        return None

    print(f"Loading data from {SAVE_PATH}...")
    df = pd.read_csv(SAVE_PATH)

    try:
        df['code_id'] = df['code_id'].astype(int)
    except Exception:
        print("Warning: Could not convert 'code_id' column to integer. Skipping filtering.")

    if 'judging_llm' in df.columns:
        initial_len = len(df)
        df = df[df['judging_llm'] != 'D']
        print(f"Excluded {initial_len - len(df)} rows where llm == 'D'.")
    else:
        print("Warning: 'llm' column not found. Skipping LLM filter.")

    print(f"Filtering data for code IDs {start_id} through {end_id}...")
    df_filtered = df[df['code_id'].between(start_id, end_id)]

    if df_filtered.empty:
        print("Filtered DataFrame is empty. Check your file contents or ID range.")
        return None

    print("Aggregating results by Code ID and Result Source...")
    df_report = df_filtered.groupby(['code_id', 'result_source'])[SCORE_COLUMNS].mean().reset_index()

    return df_report

def generate_summary_report(df_aggregated):
    """
    Takes the report aggregated by code_id and result_source, and further
    aggregates it to show the grand mean for each result_source (result_1, etc.)
    """
    if df_aggregated.empty:
        return None

    print("\nCalculating GRAND SUMMARY by Result Source...")

    df_summary = df_aggregated.groupby('result_source')[SCORE_COLUMNS].mean().reset_index()
    df_summary[SCORE_COLUMNS] = df_summary[SCORE_COLUMNS].round(2)

    return df_summary


if __name__ == "__main__":
    intermediate_report = generate_evaluation_report(start_id=1, end_id=100)

    if intermediate_report is not None:
        summary_report = generate_summary_report(intermediate_report)

        if summary_report is not None:
            print("\n" + "="*80)
            print("GRAND SUMMARY REPORT (Average Performance of Each Result Type, IDs 1-100)")
            print("="*80)
            pd.set_option('display.max_rows', None)
            pd.set_option('display.max_columns', None)
            pd.set_option('display.width', 1000)
            print(summary_report)
            print("="*80)
            print(f"Total result types summarized: {len(summary_report)}")


Loading data from /content/drive/MyDrive/full_evaluation_results6.csv...
Excluded 498 rows where llm == 'D'.
Filtering data for code IDs 1 through 100...
Aggregating results by Code ID and Result Source...

Calculating GRAND SUMMARY by Result Source...

GRAND SUMMARY REPORT (Average Performance of Each Result Type, IDs 1-100)
  result_source  business_rule_percentage  format_readability_rate  technical_jargon_rate  coverage_rate  testability_rate  atomicity_rate  faithfulness_rate
0      result_1                     98.48                     5.00                   4.96           4.77              4.62            4.92               4.89
1      result_2                     98.40                     4.99                   4.93           4.74              4.57            4.88               4.85
2      result_3                     98.14                     5.00                   4.95           4.72              4.60            4.89               4.83
3      result_4                     99.1